In [0]:
# WEEK 1 / DAY 2 — Silver Layer Standardization Framework
# Builds a config-driven mapper so adding Payer D next year = one dict entry,
# not a new notebook. This mirrors the "automated Silver layer standardization
# framework" bullet from the real project.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, lit

spark = SparkSession.builder.appName("silver_standardization").getOrCreate()
landing_path = "abfss://fileingestion@adls1eastus.dfs.core.windows.net/payers/"

# ---- STEP 1: Canonical (unified) schema — designed for Gold use cases,
#      not copied from any single payer ----
CANONICAL_CLAIMS_COLS = [
    "member_id", "claim_id", "diagnosis_code", "procedure_code",
    "service_date", "billed_amount", "allowed_amount", "provider_npi", "claim_status"
]

# ---- STEP 2: Mapping config — DATA, not code. This is the piece that scales. ----
PAYER_CLAIMS_MAPPING = {
    "payer_a": {
        "member_id": "member_id", "claim_id": "claim_id",
        "diagnosis_code": "diagnosis_code", "procedure_code": "procedure_code",
        "service_date": "service_date", "billed_amount": "billed_amount",
        "allowed_amount": "allowed_amount", "provider_npi": "provider_npi",
        "claim_status": "claim_status",
    },
    "payer_b": {
        "member_id": "MemberID", "claim_id": "ClaimNbr",
        "diagnosis_code": "DxCd", "procedure_code": "ProcCd",
        "service_date": "SvcDt", "billed_amount": "BilledAmt",
        "allowed_amount": "AllowedAmt", "provider_npi": "ProviderNPI",
        "claim_status": "ClmStatus",
    },
    "payer_c": {
        "member_id": "mbrid", "claim_id": "clmid",
        "diagnosis_code": "diagcd", "procedure_code": "proccd",
        "service_date": "dos", "billed_amount": "chgamt",
        "allowed_amount": "pdamt", "provider_npi": "npinum",
        "claim_status": "clmstat",
    },
}

def standardize_claims(bronze_df, payer_key: str):
    """Rename source columns to canonical names using the mapping config.
    Adding a new payer means adding one dict entry above — this function never changes."""
    mapping = PAYER_CLAIMS_MAPPING[payer_key]
    select_exprs = [col(source_col).alias(canonical_col)
                     for canonical_col, source_col in mapping.items()]
    df = bronze_df.select(*select_exprs)
    df = df.withColumn("source_payer", lit(payer_key))
    # enforce canonical column order + types
    df = df.select(*CANONICAL_CLAIMS_COLS, "source_payer")
    return df

# ---- STEP 3: Run it ----
bronze_a = spark.read.option("header", True).option("inferSchema", True).csv(f"{landing_path}/payer_A_claims_aetna_style.csv")
bronze_b = spark.read.option("header", True).option("inferSchema", True).csv(f"{landing_path}/payer_B_claims_uhc_style.csv")
bronze_c = spark.read.option("header", True).option("inferSchema", True).csv(f"{landing_path}/payer_C_claims_cigna_style.csv")

silver_a = standardize_claims(bronze_a, "payer_a")
silver_b = standardize_claims(bronze_b, "payer_b")
silver_c = standardize_claims(bronze_c, "payer_c")

silver_claims_unified = silver_a.unionByName(silver_b).unionByName(silver_c)

print(f"Unified Silver claims row count: {silver_claims_unified.count()}")
silver_claims_unified.printSchema()
silver_claims_unified.groupBy("source_payer").count().show()
silver_claims_unified.show(5, truncate=False)
silver_claims_unified.write.mode("overwrite").saveAsTable("silver.payer.claims_unified")
